# Estimating tumor cell-type composition from bulk gene expression

In this lab you will use single-cell RNA-sequencing data from breast tumors to
estimate the cellular makeup of about a thousand additional tumors that were only
measured in *bulk*, and then test a specific idea: when two tumor types differ in
the expression of a gene, is the gene changing inside the cells, or is the tumor
simply built from a different mix of cells?

You do not need prior Python experience. Run each code cell in order with the play
button or Shift+Enter, and read the text between cells. A few cells are marked
**Exercise** and ask you to change one line.

## 1. Background and data

**The biology.** A tumor is not a uniform mass of cancer cells. It also contains
immune cells (T cells, macrophages, B cells), fibroblasts, blood-vessel
(endothelial) cells, and others. This mixture is called the *tumor
microenvironment*. Its composition varies between patients and is linked to
prognosis and to how well treatments such as immunotherapy work.

**Two ways to measure gene expression.**
- *Single-cell RNA-seq* measures the genes used by each individual cell. It tells
  us exactly which cell types are present, but it is expensive and available for
  relatively few tumors.
- *Bulk RNA-seq* grinds up a whole tumor and measures the average gene expression
  across all of its cells at once. It is cheap and available for thousands of
  tumors, but the cell-type information is lost in the averaging.

**The two datasets.**
- A single-cell **atlas** of breast tumors (Xu et al., 2024), already labeled with
  cell types by experts. We will use it as a reference.
- Bulk RNA-seq for ~1,000 breast tumors from **TCGA** (The Cancer Genome Atlas),
  each labeled as one of two common subtypes: **IDC** (invasive ductal carcinoma,
  the most common) or **ILC** (invasive lobular carcinoma, about 10% of cases).

First, install the libraries and load the prepared data.

In [ ]:
# Colab already has most of these; this guarantees compatible versions.
!pip install -q 'seaborn>=0.13' pandas numpy scipy matplotlib
print('libraries installed')

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import nnls
from scipy.stats import mannwhitneyu
%matplotlib inline
sns.set_style('white')
plt.rcParams['figure.dpi'] = 110

RAW = 'https://raw.githubusercontent.com/bts76-pitt/bioinfo-bootcamp-2026-data/main/'
sig    = pd.read_csv(RAW + 'signature_matrix.csv', index_col=0)   # genes x cell types
viz    = pd.read_csv(RAW + 'atlas_umap.csv.gz')                   # single cells (for plots)
counts = pd.read_csv(RAW + 'atlas_lineage_counts.csv')           # cells per type in the atlas
bulk   = pd.read_csv(RAW + 'tcga_brca_bulk.csv', index_col=0)     # genes x patients
clin   = pd.read_csv(RAW + 'tcga_brca_clinical.csv')             # patient -> subtype

print('Atlas single cells shown :', f'{viz.shape[0]:,}')
print('Patient tumors (bulk)    :', f'{bulk.shape[1]:,}')
print('Cell types in reference  :', list(sig.columns))
print('Subtype counts           :', clin.subtype.value_counts().to_dict())

## 2. Exploring the single-cell atlas

The atlas contains 236,363 individual cells from breast tumors. Experts grouped
them into seven major cell types (we merged closely related subtypes, for example
all T and NK cells, to keep the analysis stable). The bar chart below shows how
many cells of each type were measured.

In [ ]:
order = counts.sort_values('n_cells', ascending=False)['lineage'].tolist()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=counts, x='n_cells', y='lineage', order=order, color='#4c72b0', ax=ax)
ax.set(xlabel='number of cells in the atlas', ylabel='', title='Cell types measured in the breast-tumor atlas')
sns.despine(); plt.tight_layout(); plt.show()

### Visualizing thousands of cells at once: the UMAP

Each cell is described by the activity of ~20,000 genes, which is far too many
dimensions to plot. **UMAP** is a method that places each cell as a point on a 2-D
map so that cells with similar gene activity sit close together. We cannot show all
236,363 cells, so the map below uses a representative sample of about 20,000. Each
point is one cell, colored by its expert-assigned type.

In [ ]:
cell_types = sorted(viz.cell_type.unique())
palette = dict(zip(cell_types, sns.color_palette('tab10', len(cell_types))))
fig, ax = plt.subplots(figsize=(8, 6.5))
for ct in cell_types:
    d = viz[viz.cell_type == ct]
    ax.scatter(d.UMAP1, d.UMAP2, s=4, color=palette[ct], label=ct, alpha=0.6)
ax.legend(markerscale=4, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
ax.set(title='UMAP of single cells, colored by cell type', xticks=[], yticks=[])
sns.despine(left=True, bottom=True); plt.tight_layout(); plt.show()

### How are cell types identified? Marker genes

Each cell type switches on a characteristic set of genes, called *marker genes*.
For example, *EPCAM* is made by epithelial (cancer) cells, *CD3D* by T cells, and
*COL1A1* by fibroblasts. Coloring the same UMAP by a single gene shows where that
gene is active: bright points have high expression. Notice that each marker lights
up a different region, which is how the cell types were assigned.

In [ ]:
markers = {'EPCAM': 'Cancer Epithelial', 'CD3D': 'T/NK Cells', 'COL1A1': 'Fibroblasts',
           'CD68': 'Myeloid', 'PECAM1': 'Endothelial', 'MS4A1': 'B/Plasma'}
fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for ax, (gene, ct) in zip(axes.ravel(), markers.items()):
    s = ax.scatter(viz.UMAP1, viz.UMAP2, s=3, c=viz[gene], cmap='viridis')
    ax.set(title=f'{gene}  (marks {ct})', xticks=[], yticks=[])
    plt.colorbar(s, ax=ax, shrink=0.7)
sns.despine(left=True, bottom=True); plt.tight_layout(); plt.show()

A more compact way to summarize markers is a **dot plot**. Each row is a cell type
and each column a marker gene. The *color* of a dot is the average expression of
that gene in that cell type, and the *size* is the percentage of cells expressing
it. A good marker forms one large, dark dot in its own cell type and small, pale
dots elsewhere.

In [ ]:
marker_genes = ['EPCAM', 'KRT8', 'CD3D', 'CD8A', 'COL1A1', 'PDGFRB',
                'CD68', 'LYZ', 'PECAM1', 'VWF', 'MS4A1', 'CD79A', 'RGS5']
rows = []
for ct in cell_types:
    sub = viz[viz.cell_type == ct]
    for g in marker_genes:
        rows.append({'cell_type': ct, 'gene': g,
                     'mean_expr': sub[g].mean(), 'pct': (sub[g] > 0).mean() * 100})
dp = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(11, 5))
sc = ax.scatter(dp.gene, dp.cell_type, s=dp.pct * 3.5, c=dp.mean_expr,
                cmap='Reds', edgecolor='0.6', linewidth=0.4)
ax.set(title='Marker genes (color = mean expression, size = % of cells expressing)')
plt.xticks(rotation=45, ha='right')
plt.colorbar(sc, label='mean expression', shrink=0.8)
plt.tight_layout(); plt.show()

**Exercise 1.** Change `gene` below to another marker and run the cell. Try
`'CD8A'` (a T-cell marker), `'VWF'` (endothelial), or `'CDH1'` (a cancer gene we
will study later). Which cells light up?

In [ ]:
gene = 'EPCAM'        # <-- change this gene, then run the cell
plt.figure(figsize=(6, 5))
s = plt.scatter(viz.UMAP1, viz.UMAP2, s=4, c=viz[gene], cmap='viridis')
plt.colorbar(s, label=f'{gene} expression'); plt.title(gene)
plt.xticks([]); plt.yticks([]); sns.despine(left=True, bottom=True)
plt.tight_layout(); plt.show()

## 3. Building a reference signature

To recover cell-type composition from bulk data, we first summarize each cell type
by its typical gene-expression profile. Averaging all cells of a type gives one
column per cell type; stacking these columns gives a **signature matrix** (genes in
rows, cell types in columns).

Using all ~20,000 genes does not work well: a handful of very highly expressed
genes dominate the later calculation and the estimates collapse onto a single cell
type. Instead we keep only **marker genes** that are specifically high in one cell
type. The prepared signature uses 50 such genes per cell type (349 genes total).

In [ ]:
print('Signature matrix:', sig.shape[0], 'marker genes x', sig.shape[1], 'cell types')
sig.head()

We can see the structure by scaling each gene to a 0-1 range (so genes with large
and small absolute values are comparable) and grouping genes by the cell type they
mark. The block-diagonal pattern shows that each cell type has a distinct set of
high-expression genes.

In [ ]:
scaled = sig.div(sig.max(axis=1).replace(0, 1), axis=0)
gene_order = np.argsort(sig.values.argmax(axis=1))
fig, ax = plt.subplots(figsize=(6, 7.5))
sns.heatmap(scaled.iloc[gene_order], cmap='magma', yticklabels=False,
            cbar_kws={'label': 'relative expression (0-1)'}, ax=ax)
ax.set(title='Reference signature (genes grouped by the cell type they mark)',
       xlabel='', ylabel='marker genes')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 4. The deconvolution model

A bulk measurement is approximately a weighted average of the cell-type profiles,
where the weights are the fractions of each cell type:

$$\text{bulk expression} \;\approx\; \text{signature} \times \text{fractions}$$

We know the bulk expression and the signature, and we solve for the fractions. Two
details make the estimate reliable:

1. **Non-negativity.** A fraction cannot be negative, so we use *non-negative least
   squares* (NNLS), which finds the fractions that best reproduce the bulk values
   while staying at or above zero.
2. **Per-gene scaling.** Before solving, we divide each gene by its largest value
   across cell types, putting every gene on a comparable 0-1 scale so that a few
   high-expression genes do not dominate.

The function below carries out these steps.

In [ ]:
def deconvolve(sig, bulk, scale_genes=True):
    # Estimate the fraction of each cell type in every bulk sample.
    shared = sorted(set(sig.index) & set(bulk.index))
    sig, bulk = sig.loc[shared], bulk.loc[shared]
    if scale_genes:
        scale = sig.max(axis=1).replace(0, 1)   # put each gene on a 0-1 scale
    else:
        scale = pd.Series(1.0, index=sig.index)
    S = sig.div(scale, axis=0).values
    out = {}
    for sample in bulk.columns:
        coef, _ = nnls(S, (bulk[sample] / scale).values)
        out[sample] = coef / coef.sum() if coef.sum() > 0 else coef
    return pd.DataFrame(out, index=sig.columns).T

print('deconvolve() defined. Shared genes with bulk:',
      len(set(sig.index) & set(bulk.index)))

### One patient at a time

Let us estimate the composition of a single tumor. The bars are the estimated
fraction of each cell type, which sum to 1.

In [ ]:
patient = bulk.columns[0]
recipe = deconvolve(sig, bulk[[patient]]).iloc[0].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=recipe.values, y=recipe.index, color='#55a868', ax=ax)
ax.set(title=f'Estimated cell-type composition of tumor {patient}', xlabel='fraction')
sns.despine(); plt.tight_layout(); plt.show()
print('Fractions sum to', round(float(recipe.sum()), 3))

### Why per-gene scaling matters

Running the same method *without* scaling lets the most highly expressed marker
genes dominate, and the estimate shifts heavily toward stromal cell types. The
comparison below shows the same tumor solved both ways.

In [ ]:
with_scale = deconvolve(sig, bulk[[patient]], scale_genes=True).iloc[0]
no_scale   = deconvolve(sig, bulk[[patient]], scale_genes=False).iloc[0]
comp = pd.DataFrame({'with scaling': with_scale, 'without scaling': no_scale})
comp.plot.bar(figsize=(8, 4), color=['#55a868', '#c44e52'])
plt.ylabel('fraction'); plt.title(f'Effect of per-gene scaling (tumor {patient})')
plt.xticks(rotation=45, ha='right'); sns.despine(); plt.tight_layout(); plt.show()

**Exercise 2.** Change `patient_index` to any number from 0 to 1091 to inspect a
different tumor's composition, then run the cell.

In [ ]:
patient_index = 0       # <-- change to any number from 0 to 1091
p = bulk.columns[patient_index]
r = deconvolve(sig, bulk[[p]]).iloc[0].sort_values(ascending=False)
sns.barplot(x=r.values, y=r.index, color='#55a868')
plt.title(f'Tumor {p}'); plt.xlabel('fraction'); sns.despine(); plt.tight_layout(); plt.show()

## 5. Estimating composition for the whole cohort

Now we apply the method to all ~1,000 tumors at once. Each row of the result is one
patient; each column is the estimated fraction of one cell type.

In [ ]:
fractions = deconvolve(sig, bulk)
print('Estimated fractions for', fractions.shape[0], 'tumors:')
fractions.round(3).head()

The distribution of each cell-type fraction across the cohort shows how variable
the tumors are. Epithelial (cancer) content is usually the largest fraction, but it
ranges widely between patients.

In [ ]:
long = fractions.melt(var_name='cell_type', value_name='fraction')
ordc = fractions.mean().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=long, x='fraction', y='cell_type', order=ordc, color='#8172b3', ax=ax)
ax.set(title='Distribution of estimated cell-type fractions across tumors', ylabel='')
sns.despine(); plt.tight_layout(); plt.show()

Cell-type fractions are not independent: when one type takes up more of a tumor,
others must take up less. The correlation heatmap below shows which cell types tend
to rise and fall together across patients (blue = move together, red = move
oppositely).

In [ ]:
corr = fractions.corr()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            square=True, cbar_kws={'label': 'correlation'}, ax=ax)
ax.set(title='Co-occurrence of cell types across tumors')
plt.tight_layout(); plt.show()

## 6. Comparing the two subtypes: ILC vs IDC

We now ask whether lobular (ILC) and ductal (IDC) tumors differ in composition.
First we attach each patient's subtype to its estimated fractions, then compare the
average makeup of each group.

In [ ]:
frac = fractions.copy()
frac['subtype'] = clin.set_index('sample')['subtype'].reindex(frac.index).values
frac = frac.dropna(subset=['subtype'])
mean_comp = frac.groupby('subtype')[list(fractions.columns)].mean()
mean_comp.plot.bar(stacked=True, colormap='tab10', figsize=(6, 5))
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
plt.ylabel('mean fraction'); plt.title('Average composition by subtype')
plt.xticks(rotation=0); sns.despine(); plt.tight_layout(); plt.show()

The averages look similar. To test each cell type properly we use the
**Mann-Whitney U test**, which compares two groups without assuming the data are
normally distributed. A small *p-value* means the difference is unlikely to be due
to chance. Because we run seven tests at once, we apply a **Bonferroni**
correction: with 7 tests, we require p < 0.05/7 (about 0.007) before calling a
difference significant.

In [ ]:
from scipy.stats import mannwhitneyu
n_tests = len(fractions.columns)
threshold = 0.05 / n_tests
results = []
for ct in fractions.columns:
    a = frac[frac.subtype == 'ILC'][ct]
    b = frac[frac.subtype == 'IDC'][ct]
    U, p = mannwhitneyu(a, b)
    effect = 1 - 2 * U / (len(a) * len(b))     # rank-biserial effect size (-1..1)
    results.append({'cell_type': ct, 'IDC_mean': b.mean(), 'ILC_mean': a.mean(),
                    'p_value': p, 'effect_size': effect,
                    'significant': p < threshold})
results = pd.DataFrame(results).sort_values('p_value')
print(f'Bonferroni threshold: p < {threshold:.4f}')
results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=frac.melt(id_vars='subtype', value_vars=list(fractions.columns),
                           var_name='cell_type', value_name='fraction'),
            x='cell_type', y='fraction', hue='subtype', hue_order=['IDC', 'ILC'],
            palette=['#bbbbbb', '#d1495b'], ax=ax)
ax.set(title='Cell-type fractions by subtype', xlabel='')
plt.xticks(rotation=30, ha='right'); sns.despine(); plt.tight_layout(); plt.show()

**Exercise 3.** Set `cell_type` to any of the seven types and run the cell to
compare its fraction between subtypes on its own.

In [ ]:
cell_type = 'Endothelial'      # <-- change to any cell type from the table above
sns.boxplot(data=frac, x='subtype', y=cell_type, order=['IDC', 'ILC'],
            hue='subtype', legend=False, palette={'IDC': '#bbbbbb', 'ILC': '#d1495b'})
U, p = mannwhitneyu(frac[frac.subtype=='ILC'][cell_type], frac[frac.subtype=='IDC'][cell_type])
plt.title(f'{cell_type}  (p = {p:.1e})'); sns.despine(); plt.tight_layout(); plt.show()

## 7. Gene expression, and the problem of composition

The compositions are broadly similar, so what distinguishes lobular cancer? We turn
to the cancer genes themselves in the bulk data. The panels below compare six
well-known breast-cancer genes between subtypes (expression is shown on a log scale
because the values span a wide range).

In [ ]:
CANCER_GENES = ['CDH1', 'ESR1', 'ERBB2', 'MKI67', 'TP53', 'PIK3CA']
sub = clin.set_index('sample')['subtype']
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, g in zip(axes.ravel(), CANCER_GENES):
    if g not in bulk.index:
        ax.set_visible(False); continue
    df = pd.DataFrame({'expr': np.log1p(bulk.loc[g]),
                       'subtype': sub.reindex(bulk.columns).values}).dropna()
    p = mannwhitneyu(df[df.subtype=='ILC'].expr, df[df.subtype=='IDC'].expr).pvalue
    sns.boxplot(data=df, x='subtype', y='expr', order=['IDC', 'ILC'], hue='subtype',
                legend=False, palette={'IDC': '#bbbbbb', 'ILC': '#d1495b'}, ax=ax)
    ax.set(title=f'{g}  (p = {p:.0e})', xlabel='', ylabel='log expression')
sns.despine(); plt.tight_layout(); plt.show()

*CDH1* stands out: it is much lower in lobular tumors. *CDH1* encodes E-cadherin, a
protein that holds epithelial cells together. Loss of E-cadherin is the defining
molecular feature of lobular carcinoma and explains its single-file growth pattern.

### Is the difference in the cells, or in the mix?

When a gene differs between subtypes in bulk data, there are two explanations:

1. the cells that make the gene genuinely changed how much they produce
   (a *cell-intrinsic* change), or
2. the tumors contain different proportions of the cells that make the gene
   (a *composition* effect).

These have very different biological meanings, and deconvolution lets us tell them
apart. First, a simple check: if low *CDH1* in ILC were only because ILC tumors had
fewer cancer cells, then the cancer-cell fraction should also be lower in ILC.

In [ ]:
chk = frac.groupby('subtype').agg(cancer_fraction=('Cancer Epithelial', 'mean'))
chk['CDH1_expression'] = [bulk.loc['CDH1', frac[frac.subtype==s].index].mean()
                          for s in chk.index]
print(chk.round(3))

The cancer-cell fraction is essentially identical in the two subtypes, yet *CDH1*
is several times lower in ILC. Same amount of cancer cells, much less *CDH1*: the
cancer cells themselves are producing less. This is a cell-intrinsic change.

We can see both patterns directly by plotting each gene against the fraction of the
cell type that makes it, colored by subtype:

- **CDH1 vs cancer-cell fraction:** at the same fraction, ILC tumors sit lower.
  The subtype, not the fraction, drives the difference (cell-intrinsic).
- **PECAM1 vs endothelial fraction:** both subtypes fall on the same upward line.
  The gene simply reports how many blood-vessel cells are present (composition).

In [ ]:
common = [s for s in frac.index if s in bulk.columns]
st = sub.reindex(common).values
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (gene, ctype, label) in zip(axes, [
        ('CDH1', 'Cancer Epithelial', 'CDH1 vs cancer-cell fraction (cell-intrinsic)'),
        ('PECAM1', 'Endothelial', 'PECAM1 vs blood-vessel fraction (composition)')]):
    for s, col in [('IDC', '#777777'), ('ILC', '#d1495b')]:
        m = st == s
        ax.scatter(frac.loc[common, ctype][m], bulk.loc[gene, common][m],
                   s=12, alpha=0.5, color=col, label=s)
    r = np.corrcoef(frac.loc[common, ctype], bulk.loc[gene, common])[0, 1]
    ax.set(xlabel=f'{ctype} fraction', ylabel=f'{gene} expression',
           title=f'{label}\nr = {r:.2f}')
    ax.legend(frameon=False)
sns.despine(); plt.tight_layout(); plt.show()

We can also quantify how much of each gene\'s variation *across all tumors* is
explained by the relevant cell-type fraction. This is the squared correlation
($r^2$) between the gene and the fraction, expressed as a percentage. A high value
means the gene is largely a readout of how many of that cell type are present
(composition); a low value means the gene varies for other reasons, including
regulation inside the cells.

In [ ]:
def pct_variation_from_composition(gene, ctype):
    # squared correlation (r^2) between a gene and a cell-type fraction, as a percent
    g = bulk.loc[gene, common].values.astype(float)
    x = frac.loc[common, ctype].values.astype(float)
    return np.corrcoef(g, x)[0, 1] ** 2 * 100

for gene, ctype in [('CDH1', 'Cancer Epithelial'), ('PECAM1', 'Endothelial')]:
    pct = pct_variation_from_composition(gene, ctype)
    verdict = 'mostly composition' if pct > 30 else 'mostly cell-intrinsic'
    print(f'{gene:7}: {pct:4.0f}% of its variation is explained by {ctype} fraction  ->  {verdict}')

**Exercise 4.** Pick a gene and the cell type that produces it, and measure how much
of its variation is composition. Try `gene='COL1A1'` with `cell_type='Fibroblasts'`
(a fibroblast gene), or `gene='ESR1'` with `cell_type='Cancer Epithelial'`. Which
genes are mostly composition, and which are mostly cell-intrinsic?

In [ ]:
gene = 'COL1A1'             # <-- change the gene
cell_type = 'Fibroblasts'   # <-- and the cell type that makes it
if gene in bulk.index:
    pct = pct_variation_from_composition(gene, cell_type)
    print(f'{gene}: {pct:.0f}% of its variation is explained by {cell_type} fraction')
else:
    print(gene, 'is not in the bulk dataset; try another gene.')

## 8. Summary and caveats

**What you did.**
- Explored a labeled single-cell atlas and saw how marker genes define cell types.
- Built a reference signature and used non-negative least squares to estimate the
  cell-type composition of ~1,000 bulk tumors.
- Compared lobular and ductal tumors and found their compositions are broadly
  similar.
- Showed that *CDH1* loss in lobular cancer is a genuine change inside the cancer
  cells (only a few percent of its variation tracks cancer-cell fraction), whereas
  a gene like *PECAM1* is largely a readout of how many blood vessels are present
  (about half of its variation tracks endothelial fraction).

**The general lesson.** A difference in bulk gene expression can reflect a real
change inside cells or merely a difference in cell-type proportions. Estimating
composition lets you separate the two, which matters whenever bulk data are
compared between groups.

**Caveats.**
- Deconvolution gives estimates, not exact counts. Accuracy depends on the
  reference: cell types absent from the reference cannot be estimated, and rare
  populations are estimated less reliably.
- The reference came from a different set of patients than the bulk tumors, so
  systematic differences between datasets can bias the estimates.
- The bulk gene list here was reduced for speed; a full analysis would use all
  genes shared with the signature.

**Extensions to try.**
- Re-run Section 6 using a different statistical threshold, or add the
  Benjamini-Hochberg correction instead of Bonferroni.
- Test every cancer gene in the panel for composition confounding, as in Exercise 4.
- Relate the estimated immune fraction to a gene linked to immunotherapy response,
  such as *CD274* (PD-L1).